In [139]:
import json
import os
import glob

import pandas as pd

In [140]:
pd.set_option("mode.copy_on_write", True)

In [141]:
datasets = [
    'hmof', 'hmofheat', 'cof', 'ppn',
    'zeolite', 'c3h6c3h8coremof', 'ch4n2'
]

In [142]:
ds_tasks = {
    ds: [i.split('-to_')[-1] for i in glob.glob(f'../experiments/evaluate/benchmark/{ds}/*')]
    for ds in datasets
}

In [170]:
pretty_labels = {
    # Models
    #'cgcnn': 'C',
    #'moformer': 'M',
    #'pmtransformer': 'P',
    #'spbnet': 'S',
    #'intellipore': 'I',
    'cgcnn': 'CGCNN',
    'moformer': 'MOFormer',
    'pmtransformer': 'PMTransformer',
    'spbnet': 'SpbNet',
    'intellipore': 'IntelliPore',
    # Datasets
    'hmof': 'hMOF',
    'hmofheat': 'hMOF',
    'cof': 'hCOF',
    'ppn': 'hPPN',
    'zeolite': 'hZEO',
    'ch4n2': 'CoRE\ 2019',
    'c3h6c3h8coremof': 'CoRE\ 2019',
    # hmofheat
    'H2-77K-2bar': 'H$_2$ heat of adsorption at 77 K and 2 bar (MOFs)',
    'CO2-298K-2.5bar': 'CO$_2$ heat of adsorption at 298 K and 2.5 bar (MOFs)',
    # hmof
    'H2-77-2': 'H$_2$ uptake at 77 K and 2 bar (MOFs)',
    'CO2-298-2.5': 'CO$_2$ uptake at 298 K and 2.5 bar (MOFs)',
    'CH4-298-2.5': 'CH$_4$ uptake at 298 K and 2.5 bar (MOFs)',
    'N2-298-0.9': 'N$_2$ uptake at 298 K and 0.9 bar (MOFs)',
    'Xe-273-10': 'Xe binary uptake at 273 K and 10 bar (MOFs)',
    'Kr-273-10': 'Kr binary uptake at 273 K and 10 bar (MOFs)',
    # cof
    'lowbar': 'CH$_4$ uptake at 298 K and 5.8 bar (COFs)',
    'highbar': 'CH$_4$ uptake at 298 K and 65 bar (COFs)',
    # ppn
    '1bar': 'CH$_4$ uptake at 298 K and 1 bar (PPNs)',
    '65bar': 'CH$_4$ uptake at 298 K and 65 bar (PPNs)',
    # zeolite
    'unitless_KH': 'CH$_4$ unitless Henry coefficient at 298 K (ZEOs)',
    'heat_of_adsorption': 'CH$_4$ isosteric heat of adsorption at 298 K (ZEOs)',
    # c3h6c3h8coremof
    'C3H6_loadings': 'C$_3$H$_6$ binary uptake at 298 K and 1 bar (MOFs)',
    'C3H8_loadings': 'C$_3$H$_8$ binary uptake at 298 K and 1 bar (MOFs)',
    'C3H8_C3H6_Selectivity_1bar': 'C$_3$H$_8$/C$_3$H$_6$ selectivity at 298 K and 1 bar (MOFs)',
    'C3H8_C3H6_Selectivity_infinite': 'C$_3$H$_8$/C$_3$H$_6$ infinite dillution selectivity at 298 K (MOFs)',
    'C3H6_Henry_298K': 'C$_3$H$_6$ log Henry coefficient at 298 K (MOFs)',
    'C3H8_Henry_298K': 'C$_3$H$_8$ log Henry coefficient at 298 K (MOFs)',
    'TSN_S_1Bar': r'$N_\mathrm{C_3H_8} \times \log(S_\mathrm{C_3H_8/C_3H_6})$ at 298 K and 1 bar (MOFs)',
    # ch4n2
    'ch4n2ratio-0.1bar': 'CH$_4$/N$_2$ selectivity and 298 K and 0.1 bar (MOFs)',
    'ch4n2ratio-1bar': 'CH$_4$/N$_2$ selectivity and 298 K and 1 bar (MOFs)',
    'ch4n2ratio-10bar': 'CH$_4$/N$_2$ selectivity and 298 K and 10 bar (MOFs)',
    'ch4uptake-0.1bar': 'CH$_4$ binary uptake at 298 K and 0.1 bar (MOFs)',
    'ch4uptake-1bar': 'CH$_4$ binary uptake at 298 K and 1 bar (MOFs)',
    'ch4uptake-10bar': 'CH$_4$ binary uptake at 298 K and 10 bar (MOFs)',
    'n2uptake-0.1bar': 'N$_2$ binary uptake at 298 K and 0.1 bar (MOFs)',
    'n2uptake-1bar': 'N$_2$ binary uptake at 298 K and 1 bar (MOFs)',
    'n2uptake-10bar': 'N$_2$ binary uptake at 298 K and 10 bar (MOFs)',
}

<>:19: SyntaxWarning: invalid escape sequence '\ '
<>:20: SyntaxWarning: invalid escape sequence '\ '
<>:19: SyntaxWarning: invalid escape sequence '\ '
<>:20: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_251107/305442408.py:19: SyntaxWarning: invalid escape sequence '\ '
  'ch4n2': 'CoRE\ 2019',
/tmp/ipykernel_251107/305442408.py:20: SyntaxWarning: invalid escape sequence '\ '
  'c3h6c3h8coremof': 'CoRE\ 2019',


In [183]:
def filter_spbnet_df(df):
    df.rename(columns={'Unnamed: 0': 'Model'}, inplace=True)
    df.dropna(inplace=True)
    df = df[~df.Metric.isin(['MSE', 'MAE (Relative Reduction)', 'MAE'])]
    df.set_index('Model', inplace=True)
    df.drop(columns='Metric', inplace=True)
    return df

In [184]:
def load_json(path):
    with open(path) as fhand:
        return json.load(fhand)

In [185]:
#def extract_results(dataset, task, save=False, metric='test_R2Score'):
def extract_results(dataset, metric='test_R2Score'):

    #train_size = 5000
    #test_size = 1000
    #if dataset == 'hmofheat':
    #    test_size = 700
    #if dataset == 'ch4n2':
    #    train_size = 3657
    #    test_size = 458
    #if dataset == 'c3h6c3h8coremof':
    #    train_size = 1354
    #    test_size = 170
        
    size_key = 'null' if dataset != 'hmofheat' else '5000'
    run_key = '0'
    ckpt_path = 'from_experiments_pretrain_early_pretrain_lightning_logs_version_0_checkpoints_best.ckpt-to_'

    results = {}
    
    for task in ds_tasks[dataset]:
        json_path = f'../experiments/evaluate/benchmark/{dataset}/{ckpt_path}{task}/results.json'
        lbl = pretty_labels[task]
        results[lbl] = spbnet_dfs[dataset][task].to_dict()
        results[lbl]['intellipore'] = float(load_json(json_path)[size_key][run_key][metric])
        results[lbl]['Dataset'] = pretty_labels[dataset]

    return results

In [186]:
# Convert excel to dataframe
xl = pd.read_excel('/home/asarikas/databases/SpbNet/41467_2026_69245_MOESM3_ESM.xlsx', sheet_name=None)
spbnet_dfs = {ds: filter_spbnet_df(xl[ds]) for ds in datasets}

In [187]:
res_dict = {}

In [188]:
for ds in datasets:
    res_dict[ds] = pd.DataFrame(extract_results(ds, metric='test_R2Score')).T

In [189]:
df_res = pd.concat(res_dict.values(), axis=0)
df_res = df_res.rename(mapper=pretty_labels, axis=1)

In [190]:
df_res.dtypes

CGCNN            object
MOFormer         object
PMTransformer    object
SpbNet           object
IntelliPore      object
Dataset          object
dtype: object

In [191]:
col = df_res.pop('Dataset')

In [192]:
df_res = df_res.astype('float')

In [193]:
df_res.dtypes

CGCNN            float64
MOFormer         float64
PMTransformer    float64
SpbNet           float64
IntelliPore      float64
dtype: object

## Latex

In [195]:
# Highlight the best model with bold
df_latex = df_res.copy()

for idx, row in df_latex.iterrows():
    best_col = row.idxmax()
    #best_col = row.idxmin()
    best_val = row[best_col]
    
    df_latex.loc[idx, best_col] = (
        f"\\textbf{{{best_val:.3f}}}"
    )


# Insert dataset column
df_latex.insert(0, 'Dataset', col)
#latex = df_latex.to_latex(float_format='{:.3f}'.format)
#latex = latex.astype('string')
#latex.insert(0, 'Dataset', col)
latex = df_latex.to_latex(float_format='{:.3f}'.format)

print(latex)

\begin{tabular}{llrrrll}
\toprule
 & Dataset & CGCNN & MOFormer & PMTransformer & SpbNet & IntelliPore \\
\midrule
CO$_2$ uptake at 298 K and 2.5 bar (MOFs) & hMOF & 0.605 & 0.563 & 0.886 & \textbf{0.944} & 0.943 \\
CH$_4$ uptake at 298 K and 2.5 bar (MOFs) & hMOF & 0.667 & 0.608 & 0.853 & 0.924 & \textbf{0.960} \\
N$_2$ uptake at 298 K and 0.9 bar (MOFs) & hMOF & 0.634 & 0.562 & 0.777 & 0.874 & \textbf{0.906} \\
Xe binary uptake at 273 K and 10 bar (MOFs) & hMOF & 0.697 & 0.675 & 0.901 & 0.926 & \textbf{0.946} \\
Kr binary uptake at 273 K and 10 bar (MOFs) & hMOF & 0.733 & 0.736 & 0.842 & 0.856 & \textbf{0.899} \\
H$_2$ uptake at 77 K and 2 bar (MOFs) & hMOF & 0.621 & 0.627 & 0.932 & 0.975 & \textbf{0.985} \\
CO$_2$ heat of adsorption at 298 K and 2.5 bar (MOFs) & hMOF & 0.812 & 0.755 & 0.906 & 0.903 & \textbf{0.909} \\
H$_2$ heat of adsorption at 77 K and 2 bar (MOFs) & hMOF & 0.798 & 0.711 & 0.895 & 0.905 & \textbf{0.926} \\
CH$_4$ uptake at 298 K and 5.8 bar (COFs) & hCOF & 0.516 &

/tmp/ipykernel_251107/3188374821.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '\textbf{0.944}' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_latex.loc[idx, best_col] = (
/tmp/ipykernel_251107/3188374821.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '\textbf{0.960}' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_latex.loc[idx, best_col] = (


## Markdown

In [196]:
import re

subscripts = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")

def latex_subscript_to_unicode(s):
    return re.sub(
        r"\$_(\d+)\$",
        lambda m: m.group(1).translate(subscripts),
        s
    )

In [82]:
df_md = df_res.copy()

# Format everything first
df_md = df_md.map(lambda x: f"{x:.3f}")

# Bold winners using the ORIGINAL numeric dataframe
for idx, row in df_res.iterrows():
    best_col = row.idxmax()
    df_md.loc[idx, best_col] = f"**{row[best_col]:.3f}**"

df_md.index = df_md.index.map(latex_subscript_to_unicode)

markdown = df_md.to_markdown(disable_numparse=True)
print(markdown)

|                                                                                     | CGCNN   | MOFormer   | PMTransformer   | SpbNet    | IntelliPore   |
|:------------------------------------------------------------------------------------|:--------|:-----------|:----------------|:----------|:--------------|
| CO₂ uptake at 298 K and 2.5 bar (MOFs)                                              | 0.605   | 0.563      | 0.886           | **0.944** | 0.943         |
| CH₄ uptake at 298 K and 2.5 bar (MOFs)                                              | 0.667   | 0.608      | 0.853           | 0.924     | **0.960**     |
| N₂ uptake at 298 K and 0.9 bar (MOFs)                                               | 0.634   | 0.562      | 0.777           | 0.874     | **0.906**     |
| Xe binary uptake at 273 K and 10 bar (MOFs)                                         | 0.697   | 0.675      | 0.901           | 0.926     | **0.946**     |
| Kr binary uptake at 273 K and 10 bar (MOFs)             